# OpenPOPCON SPARC example

SPARC is the compact, high-field D-T tokamak being built by Commonwealth
Fusion Systems and MIT: 1.85 m major radius, 12.2 T, 8.7 MA. Machine
parameters are from Creely et al., "Overview of the SPARC tokamak",
J. Plasma Phys. 86 (2020) 865860502.

This example uses parabolic profiles rather than a gEQDSK, which keeps `R`,
`a`, `kappa`, `delta` and `I_P` free for `POPCON_scan` to vary.

In [ ]:
import numpy as np
import openpopcon as op

## Setup and run

`settingsfile` holds the machine and algorithm settings, `plotsettingsfile`
the contour levels and axes. Both are read from this directory.

The numerics are compiled with numba the first time they run, so the first
solve in a fresh kernel takes a few seconds longer than the rest.

In [ ]:
settingsfile = "./POPCON_input_example.yaml"
plotsettingsfile = "./plotsettings.yml"

pc = op.POPCON(settingsfile=settingsfile, plotsettingsfile=plotsettingsfile)
pc.run_POPCON()

## Plotting

This example's plotsettings draw the volume-averaged density on the y axis
rather than the Greenwald fraction.

In [ ]:
fig, ax = pc.plot()

### A point on the plot

The numbers behind one grid point, picked by Greenwald fraction and
volume-averaged ion temperature.

In [ ]:
i = int(np.abs(pc.output.n_G_frac.values - 0.37).argmin())
j = int(np.abs(pc.output.T_i_avg.values - 7.3).argmin())
point = pc.output.isel(n_index=i, T_index=j)

print(f"n/n_G   = {float(point.n_G_frac):.2f}")
print(f"<T_i>   = {float(point.T_i_avg):.1f} keV")
print(f"P_fus   = {float(point.Pfusion):.1f} MW")
print(f"P_aux   = {float(point.Paux):.1f} MW")
print(f"Q       = {float(point.Q):.2f}")
print(f"beta_N  = {float(point.betaN):.2f}")

## Scoping a single operating point

`single_point` solves one density/temperature pair and shows the profiles
behind it, which is the quickest way to see why a point on the POPCON sits
where it does.

In [ ]:
pc.single_point(n_G_frac=0.37, Ti_av=7.3)

## Scanning the major radius against the field

`POPCON_scan` runs a full POPCON at every combination of two machine
parameters. The ranges come from the `scan:` block at the bottom of the
settings file.

The rows scan `R_0`, which is an alias for `R`: the settings file gives `R`,
and each cell replaces it with the scanned `R_0`. The current is held fixed,
so a larger machine here also has a higher q and a lower Greenwald density.

In [ ]:
sc = op.POPCON_scan(settingsfile=settingsfile, plotsettingsfile=plotsettingsfile)
sc.run_scan()

In [ ]:
fig, axs = sc.plot()

`plot_metric` reduces each cell to one number so the trend across the scan
reads at a glance. Anything in the output works, with any reduction.

In [ ]:
fig, ax = sc.plot_metric("Pfusion", reduce="max")

### Scanning from Python instead of the settings file

Passing `scan=` overrides the block in the settings file. Each axis takes
either an explicit list of values or a `min`/`max`/`N` range.

In [ ]:
sc2 = op.POPCON_scan(
    settingsfile=settingsfile,
    plotsettingsfile=plotsettingsfile,
    scan={
        "rows": ("I_P", {"min": 7.5, "max": 9.9, "N": 3}),
        "cols": ("H_fac", [0.8, 1.0, 1.2]),
    },
)
sc2.run_scan()
fig, axs = sc2.plot()

## Saving and reloading a scan

`write_output` saves the whole scan: the base settings, the scan
specification, the combined arrays and the grid plot.

In [ ]:
sc.write_output(name="sparc_scan", archive=False, overwrite=True)

back = op.POPCON_scan.read_output("sparc_scan")
print(back.shape, back.row.parameter, back.col.parameter)